# TD 2 — Corrigé

**Analyse des données — L3 Économie**

> **Note pour le chargé de TD.** Les chiffres ci-dessous proviennent de l'exécution sur l'Enquête Emploi 2024. Ils sont à revérifier si le millésime change.
>
> Trois cellules de contrôle ont été ajoutées à cette version : le croisement `HALOR` × `ACTEU`, le profil par diplôme sans la non-réponse, et la liste des tranches d'âge effectivement présentes dans le profil. **Exécutez-les avant la séance** : deux des commentaires qui suivent dépendent de ce qu'elles affichent.

Page du fichier : <https://www.insee.fr/fr/statistiques/8632441>

## Partie 1 — Charger et inspecter

In [ ]:
import pandas as pd
import numpy as np

URL = "https://www.insee.fr/fr/statistiques/fichier/8632441/FD_EEC_2024.parquet"

df = pd.read_parquet(URL)
print(df.shape)
df.head()

**Réponse 1.** 353 420 lignes, 83 colonnes. Une ligne décrit **une personne interrogée**, pas un ménage, pas un logement.

C'est le premier réflexe, et il commande tout le reste : chaque ligne est un individu, et chacun représente un certain nombre de Français. D'où la pondération, partie 2.

In [ ]:
df["ACTEU"].value_counts(dropna=False).sort_index()

In [ ]:
pd.crosstab(df["ACTEU"], df["AGE6"], dropna=False)

**Réponse 2.** Le dictionnaire indique pour `ACTEU` : *champ — personnes de 15 ans ou plus*.

Le croisement le confirme : la tranche `00` (0 à 14 ans) n'a **aucune** valeur d'`ACTEU`. Ces lignes ne sont pas des non-réponses, elles sont **hors champ** : la question ne leur a pas été posée.

C'est la distinction centrale de la séance 3. Une personne hors champ n'aurait pas dû répondre ; une non-réponse aurait dû. Les traiter de la même façon est une faute.

> Retenez ce croisement. Il ressert à la réponse 10, où les 0-14 ans ne produisent pas un `NaN` bien visible mais disparaissent purement et simplement du tableau.

## Partie 2 — Le taux de chômage

In [ ]:
n = df["ACTEU"].value_counts()

taux_brut = 100 * n["2"] / (n["1"] + n["2"])
print("Taux de chomage, sans ponderation :", round(taux_brut, 2), "%")

**Réponse 3.** Le taux non pondéré vaut **7,95 %**.

Le taux publié par l'INSEE pour 2024 est de l'ordre de **7,4 %**. Un demi-point d'écart, ce qui est considérable pour un indicateur dont l'INSEE annonce une précision de ± 0,2 point. L'écart ne vient d'aucune erreur de calcul.

In [ ]:
p = df.groupby("ACTEU")["EXTRIAN"].sum()

taux_pondere = 100 * p["2"] / (p["1"] + p["2"])
print("Taux de chomage, pondere :", round(taux_pondere, 2), "%")

**Réponse 4.** Le taux pondéré vaut **7,44 %**, à un dixième du chiffre publié. L'écart avec la version non pondérée est de **0,51 point**.

L'explication tient à la construction de l'enquête. Les quelque 90 000 personnes interrogées chaque trimestre ne sont pas un échantillon miniature de la France : certaines catégories sont volontairement sur-représentées pour que les estimations restent précises. Le poids `EXTRIAN` corrige cela. Il est redressé après correction de la non-réponse, puis calé sur la structure de la population par sexe et âge.

**Compter les lignes décrit l'échantillon. Sommer les poids décrit la population.**

C'est la première fois que les étudiants peuvent *vérifier* cette affirmation plutôt que la croire : le chiffre publié sert de contrôle.

In [ ]:
print("Somme des poids :", f"{df['EXTRIAN'].sum():,.0f}".replace(",", " "))

**Réponse 5.** La somme des poids vaut **55 435 998**, soit **55,4 millions de personnes**.

Ce total reconstitue la population de 15 ans ou plus vivant en ménage ordinaire. C'est un ordre de grandeur plausible pour une population française de quelque 68 millions d'habitants dont environ un cinquième a moins de 15 ans.

Trois exclusions à signaler : 15 ans ou plus, ménages ordinaires, donc hors foyers, hôpitaux et prisons. Le total ne doit pas être confondu avec la population totale de la France.

Ce contrôle vaut d'être fait systématiquement : si la somme des poids ne ressemble pas à la population attendue, quelque chose ne va pas dans le filtrage.

## Partie 3 — Deux chiffres officiels

In [ ]:
tableau = pd.crosstab(
    df["ACTEU"],
    df["OFFICC"],
    values=df["EXTRIAN"],
    aggfunc="sum"
)

(tableau / 1000).round(0)     # en milliers

**Réponse 6.** Les quatre cases, en milliers de personnes :

| en milliers | Inscrit | Non inscrit | **Total** |
|---|---:|---:|---:|
| **En emploi** | 2 501 | 26 521 | 29 022 |
| **Chômeur BIT** | 1 787 | 544 | 2 331 |
| **Inactif** | 1 560 | 21 835 | 23 395 |
| **Total** | **5 848** | 48 900 | 54 748 |

Le total du tableau, 54 748 milliers, est inférieur à la somme des poids (55 436). L'écart de 688 milliers correspond aux non-réponses et au hors champ : `OFFICC` ne concerne que les 15-74 ans.

**Les deux chiffres à retenir : 2,33 millions de chômeurs BIT, 5,85 millions d'inscrits à France Travail.** Deux fois et demie plus d'inscrits que de chômeurs au sens du BIT.

Chaque case correspond à une situation réelle :

- **Chômeur BIT inscrit, 1 787** : le cas attendu, présent dans les deux statistiques. Il ne représente que **77 %** des chômeurs BIT.
- **Chômeur BIT non inscrit, 544**, soit **23 %** d'entre eux. Ils cherchent activement et sont disponibles, mais ne se sont pas inscrits : jeune diplômé passant par son réseau, personne sans droits à indemnisation.
- **En emploi et inscrit, 2 501.** C'est la case la plus frappante : **plus nombreuse que les chômeurs BIT inscrits**. Elle rassemble les personnes en contrat court ou à temps partiel qui restent inscrites pour trouver mieux. Le BIT les compte en emploi dès une heure travaillée dans la semaine.
- **Inactif et inscrit, 1 560** : inscrits, mais ni disponibles ni en recherche active la semaine de référence. Formation, santé, découragement.

Autrement dit, **moins d'un inscrit sur trois est un chômeur au sens du BIT**.

**Réponse 7.** Les deux chiffres ne sont pas contradictoires : ils répondent à des questions différentes.

Le **chômage BIT** est une mesure du marché du travail, définie par une norme internationale, et c'est le seul qui permette de comparer la France à l'Allemagne.

Les **demandeurs d'emploi inscrits** relèvent d'une logique administrative : ils dénombrent les personnes suivies par le service public de l'emploi, ce qui est le bon périmètre pour dimensionner un budget d'indemnisation ou un dispositif d'accompagnement.

Utiliser l'un pour la question de l'autre est l'erreur classique du débat public.

> **Une précision à faire, puisque la question oppose les deux sources.** Les 5 848 milliers que vous venez de calculer sont des personnes qui **déclarent** être inscrites, dans une enquête. Ce n'est pas le dénombrement administratif publié par France Travail, qui compte des dossiers et non des déclarations, et qui répartit les inscrits en catégories A à E. Les deux nombres sont du même ordre sans être le même nombre, et l'écart tient à la façon dont chacun est produit.

## Pour ceux qui ont terminé

In [ ]:
halo = df.groupby("HALOR")["EXTRIAN"].sum()
(halo / 1000).round(0)

**Réponse 8.** Le halo représente **1 939 milliers de personnes**, contre **2 331 milliers** de chômeurs BIT, soit **83 %** du nombre de chômeurs.

Si on les comptait comme chômeurs, le taux passerait de 7,4 % à **12,8 %**.

Ces personnes ne sont **dans aucun des deux** : ni au numérateur, ni au dénominateur. Au sens du BIT elles sont inactives, puisqu'elles ne remplissent pas les trois critères simultanément.

C'est pourquoi le taux de chômage ne suffit pas à décrire l'éloignement du marché du travail, et pourquoi l'INSEE publie le halo à côté.

**Réponse 9.** Le halo est **défini** comme un sous-ensemble des inactifs : personnes sans emploi, qui en souhaitent un, mais que les critères du BIT classent hors du chômage. La question n'a donc de sens que pour elles.

### Une anomalie apparente, et comment la trancher

La somme des poids par modalité de `HALOR` donne 1 939 + 53 497 = **55 436**, soit la totalité de l'échantillon. Or le dictionnaire annonce un champ restreint aux inactifs, qui ne sont que 23 395 milliers.

**Deux lectures sont possibles, et elles n'ont pas les mêmes conséquences.**

1. La modalité `2` signifie « n'appartient pas au halo » et joue le rôle d'un **résiduel attribué à tout le monde**. Le champ documenté porterait alors sur la modalité `1` seule. Ce serait une convention de codage, pas une erreur.
2. La variable est réellement renseignée au-delà de son champ documenté, et la documentation ne décrit pas le fichier.

**Ne tranchez pas sans regarder.** Le croisement suivant décide.

In [ ]:
# Le halo est-il bien contenu dans les inactifs ?
verif = pd.crosstab(df["HALOR"], df["ACTEU"],
                    values=df["EXTRIAN"], aggfunc="sum")
print((verif / 1000).round(0))

ligne = verif.loc["1"]                       # les personnes du halo
part  = 100 * ligne.get("3", 0) / ligne.sum()

print()
print("part des HALOR=1 qui sont inactifs (ACTEU=3) :", round(part, 1), "%")

**Comment lire ce croisement.**

Si la ligne `HALOR = 1` est **entièrement** dans la colonne `ACTEU = 3`, c'est la première lecture qui est la bonne. Le halo est bien un sous-ensemble des inactifs, exactement comme le dit la réponse 9, et seule la modalité résiduelle `2` est généralisée à tout l'échantillon. Il n'y a alors **aucune erreur de documentation à signaler** : il y a une convention de codage qu'il faut avoir lue.

Si des personnes en emploi ou au chômage apparaissent en `HALOR = 1`, la seconde lecture s'impose, et l'écart avec la documentation est réel.

**Dans les deux cas, la leçon est la même, et c'est elle qu'il faut faire ressortir.** Un dictionnaire décrit une intention. Ce qui se trouve dans le fichier reste à vérifier, et la vérification tient en une ligne de code. Le bon réflexe n'est pas de faire confiance au champ annoncé, ni de crier à l'erreur : c'est de restreindre soi-même le calcul et de regarder ce que cela change.

> **Pourquoi cette prudence.** Affirmer devant un amphi que l'INSEE documente mal son fichier engage davantage qu'une remarque technique. Sur les données que nous avons, l'explication par la modalité résiduelle est la plus simple, et c'est elle qu'il faut écarter en premier avant d'accuser la source.

### Décliner le taux

In [ ]:
def taux_par(variable):
    p = df.groupby([variable, "ACTEU"])["EXTRIAN"].sum().unstack(fill_value=0)
    return (100 * p["2"] / (p["1"] + p["2"])).round(2)


print(taux_par("AGE6"))
print()
print(taux_par("DIP7"))

In [ ]:
# Deux controles que le tableau ci-dessus ne fait pas apparaitre.

# 1. Les 0-14 ans ne renvoient pas NaN : ils disparaissent du resultat.
print("tranches d'age dans le fichier :", sorted(df["AGE6"].dropna().unique()))
print("tranches dans le profil        :", sorted(taux_par("AGE6").index))

# 2. La modalite 9 de DIP7 est une non-reponse, pas un niveau de diplome.
profil = taux_par("DIP7")
print()
print("profil par diplome, sans la non-reponse :")
print(profil.drop("9", errors="ignore"))

**Réponse 10.** Deux profils, et chacun cache un piège de lecture avant même l'interprétation économique.

**Par âge.** 15-24 ans : **18,8 %** ; 25-49 ans : 6,7 % ; 50-64 ans : 5,0 % ; 65-89 ans : 3,3 %.

Le taux des jeunes est près de trois fois celui des 25-49 ans. Mais attention à la lecture : à cet âge, la majorité de la classe d'âge est en études, donc inactive et **absente du dénominateur**. Dire « un jeune sur cinq est au chômage » est faux. C'est un jeune *actif* sur cinq, et les actifs sont minoritaires à cet âge.

La tranche 90 ans ou plus renvoie `NaN` : aucun actif, donc division par zéro. Un résultat vide est parfois la bonne réponse.

**Et la tranche `00` ?** Elle n'apparaît nulle part, pas même en `NaN`. `groupby` écarte les lignes dont la clé de regroupement est manquante, et `ACTEU` est vide pour tous les 0-14 ans. Ils ne laissent donc **aucune trace** dans le tableau.

C'est la suite directe de la réponse 2, et le point le plus important de cette cellule : **un hors champ ne produit pas toujours un trou visible. Il peut produire une ligne qui n'existe pas.** Seule la comparaison entre les modalités du fichier et celles du résultat le révèle, et c'est ce que fait la cellule de contrôle.

**Par diplôme.** bac+5 : 4,7 % ; bac+3/4 : 5,9 % ; **bac+2 : 4,6 %** ; bac : 8,9 % ; CAP-BEP : 7,3 % ; **BEPC : 15,0 %** ; aucun diplôme : 13,2 %.

Le gradient n'est **pas monotone**, contrairement à ce qu'on attend. Deux anomalies apparentes :

- le **bac+2** fait mieux que le bac+5 et que le bac+3/4. Ce sont les BTS et DUT, diplômes professionnalisants à l'insertion rapide ;
- le **BEPC seul** fait pire que l'absence de diplôme, ce qui invite à la prudence : l'effectif est faible et la composition par âge très particulière.

La coupure nette se situe entre le supérieur et le reste, non le long d'une échelle régulière. C'est un bon avertissement avant l'équation de Mincer de la séance 7 : le rendement de l'éducation n'est pas une droite.

**Et la modalité 9 ?** Elle sort du `groupby` comme les autres, et elle affiche un taux. Ce n'est pas un niveau de diplôme : c'est la **non-réponse**, un groupe qui mélange tous les niveaux. Le nombre affiché n'a aucune interprétation sur l'échelle des diplômes, et il ne se lit pas dans le gradient.

Le retirer est une décision, et elle se déclare : la seconde cellule de contrôle affiche le profil sans elle, et le texte doit dire combien de personnes ont été écartées.

> **À faire remarquer.** Le tableau des erreurs fréquentes, plus bas, cite « inclure la modalité 9 dans un profil ». Le code de ce corrigé la sort. C'est volontaire : les étudiants doivent d'abord la **voir** apparaître pour apprendre à la retirer.

## Erreurs fréquentes

| Erreur | Symptôme | Ce qu'elle enseigne |
|---|---|---|
| Oublier `EXTRIAN` | taux éloigné du chiffre publié | Compter les lignes décrit l'échantillon |
| Traiter les 0-14 ans comme des manquants | dénominateur gonflé | Hors champ n'est pas non-réponse |
| Croire qu'un hors champ laisse un `NaN` | une tranche d'âge disparue, sans message | Un groupe absent du résultat ne se voit qu'en comptant |
| Inclure la modalité 9 dans un profil | une catégorie « non réponse » apparaît | Une non-réponse n'est pas une modalité |
| Conclure à une erreur de documentation sans croiser | une accusation portée sur une ligne de sortie | Vérifier l'explication la plus simple d'abord |
| Lire le taux des jeunes comme une part | « un jeune sur quatre au chômage » | Le dénominateur est la population active |
| Opposer BIT et France Travail | « l'un des deux ment » | Deux définitions, deux usages |
| Confondre inscrits déclarés et inscrits administratifs | deux chiffres du même ordre traités comme identiques | La source produit le chiffre autant que le concept |

---

Supports, données et corrigés : `stefaniamarcassa.github.io/analyse_des_donnees`